In [43]:
import pandas as pd
import requests

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [44]:
import pandas as pd

# Lista oficial de municípios da Paraíba
url_municipios = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"

municipios = pd.read_csv(url_municipios)

# Filtrar apenas os municípios da Paraíba (código UF = 25)
municipios = municipios[municipios["codigo_uf"] == 25].copy()

# Manter apenas as informações necessárias
municipios = municipios[
    ["codigo_ibge", "nome"]
].reset_index(drop=True)

print("Municípios da Paraíba carregados com sucesso!")
print("Total:", len(municipios))

municipios.head()

Municípios da Paraíba carregados com sucesso!
Total: 223


,codigo_ibge,nome
0,2500106,Água Branca
1,2500205,Aguiar
2,2500304,Alagoa Grande
3,2500403,Alagoa Nova
4,2500502,Alagoinha


In [45]:
import zipfile
import os

arquivo_zip = "/content/2_1_Populacao_residente_Area_territorial_Densidade_demografica_xlsx.zip"
pasta_destino = "/content/dados_ibge"

os.makedirs(pasta_destino, exist_ok=True)

with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
    zip_ref.extractall(pasta_destino)

print("Arquivo descompactado com sucesso!")

arquivos = os.listdir(pasta_destino)
print(f"Total de arquivos encontrados: {len(arquivos)}")

[x for x in arquivos if "PB" in x]

Arquivo descompactado com sucesso!
Total de arquivos encontrados: 29


['tabela_2_1_PB.xlsx']

In [46]:
arquivo_pb = "/content/dados_ibge/tabela_2_1_PB.xlsx"

# Primeiro vamos visualizar a planilha como ela veio do IBGE
dados_pb = pd.read_excel(arquivo_pb, header=None)

print("Dimensão da planilha:", dados_pb.shape)

dados_pb.head(15)

Dimensão da planilha: (229, 4)


,0,1,2,3
0,"Tabela 4714 - População Residente, Área territ...",NaN,NaN,NaN
1,Unidade da Federação e Município,Ano x Variável,NaN,NaN
2,NaN,2022,NaN,NaN
3,NaN,População residente (Pessoas),Área da unidade territorial (Quilômetros quadr...,Densidade demográfica (Habitante por quilômetr...
4,Paraíba,3974495,56467,70.39
5,Água Branca (PB),9335,242,38.63
6,Aguiar (PB),5003,352,14.23
7,Alagoa Grande (PB),26062,322,80.92
8,Alagoa Nova (PB),21013,128,163.87
9,Alagoinha (PB),13725,111,123.25


In [47]:
# Seleciona somente as linhas dos municípios
municipios_pb = dados_pb.iloc[5:].copy()

# Define nomes simples para as colunas
municipios_pb.columns = [
    "municipio",
    "populacao",
    "area_km2",
    "densidade_demografica"
]

# Remove "(PB)" do nome dos municípios
municipios_pb["municipio"] = (
    municipios_pb["municipio"]
    .astype(str)
    .str.replace(r"\s*\(PB\)$", "", regex=True)
    .str.strip()
)

# Converte as colunas numéricas
municipios_pb["populacao"] = pd.to_numeric(
    municipios_pb["populacao"], errors="coerce"
)

municipios_pb["area_km2"] = pd.to_numeric(
    municipios_pb["area_km2"], errors="coerce"
)

municipios_pb["densidade_demografica"] = pd.to_numeric(
    municipios_pb["densidade_demografica"], errors="coerce"
)

# Remove eventuais linhas finais que não sejam municípios
municipios_pb = municipios_pb.dropna(subset=["populacao"])

# Reorganiza o índice
municipios_pb = municipios_pb.reset_index(drop=True)

print("Quantidade de municípios:", len(municipios_pb))

municipios_pb.head(10)

Quantidade de municípios: 223


,municipio,populacao,area_km2,densidade_demografica
0,Água Branca,9335.0,242.0,38.63
1,Aguiar,5003.0,352.0,14.23
2,Alagoa Grande,26062.0,322.0,80.92
3,Alagoa Nova,21013.0,128.0,163.87
4,Alagoinha,13725.0,111.0,123.25
5,Alcantil,5578.0,310.0,18.00
6,Algodão de Jandaíra,2953.0,223.0,13.26
7,Alhandra,21713.0,184.0,118.02
8,São João do Rio do Peixe,17964.0,476.0,37.72
9,Amparo,2234.0,122.0,18.30


In [48]:
print("Total de municípios:", len(municipios_pb))
print("Valores ausentes:")
print(municipios_pb.isnull().sum())

print("\nPopulação total dos municípios:")
print(int(municipios_pb["populacao"].sum()))

Total de municípios: 223
Valores ausentes:
municipio                0
populacao                0
area_km2                 0
densidade_demografica    0
dtype: int64

População total dos municípios:
3974495


In [49]:
# Ajusta os tipos numéricos
municipios_pb["populacao"] = municipios_pb["populacao"].astype(int)
municipios_pb["area_km2"] = municipios_pb["area_km2"].astype(int)

# Salva a base tratada
caminho_saida = "/content/populacao_municipios_pb_2022.csv"

municipios_pb.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo criado com sucesso!")
print(caminho_saida)

Arquivo criado com sucesso!
/content/populacao_municipios_pb_2022.csv


In [50]:
import requests
import pandas as pd

url = "https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/9324/p/2022"

try:
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    dados = r.json()

    print("Conexão realizada!")
    print("Quantidade de registros:", len(dados))
    print(dados[:2])

except Exception as erro:
    print("Erro ao acessar:", erro)

Conexão realizada!
Quantidade de registros: 1
[{'NC': 'Nível Territorial (Código)', 'NN': 'Nível Territorial', 'MC': 'Unidade de Medida (Código)', 'MN': 'Unidade de Medida', 'V': 'Valor', 'D1C': 'Município (Código)', 'D1N': 'Município', 'D2C': 'Variável (Código)', 'D2N': 'Variável', 'D3C': 'Ano (Código)', 'D3N': 'Ano'}]


In [51]:
url = "https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2025/v/9324/p/2022"

r = requests.get(url, timeout=30)
r.raise_for_status()

dados_pb = r.json()

print("Quantidade de registros:", len(dados_pb))
print(dados_pb[:3])

Quantidade de registros: 1
[{'NC': 'Nível Territorial (Código)', 'NN': 'Nível Territorial', 'MC': 'Unidade de Medida (Código)', 'MN': 'Unidade de Medida', 'V': 'Valor', 'D1C': 'Município (Código)', 'D1N': 'Município', 'D2C': 'Variável (Código)', 'D2N': 'Variável', 'D3C': 'Ano (Código)', 'D3N': 'Ano'}]


In [52]:
import zipfile
import pandas as pd

arquivo_zip = "/content/Agregados_por_setores_basico_BR_20260520.zip"

with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
    print("Arquivos dentro do ZIP:")
    print(zip_ref.namelist())

Arquivos dentro do ZIP:
['Agregados_por_setores_basico_BR.csv']


In [53]:
with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
    nome_csv = zip_ref.namelist()[0]

    with zip_ref.open(nome_csv) as arquivo:
        setores = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            low_memory=False
        )

print("Total de registros no Brasil:", len(setores))

print("\nColunas encontradas:")
print(setores.columns.tolist())

Total de registros no Brasil: 468099

Colunas encontradas:
['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'v0001', 'v0002', 'v0003', 'v0004', 'v0005', 'v0006', 'v0007', 'v0008', 'v0009']


In [54]:
# Filtrar somente os setores censitários da Paraíba
setores_pb = setores[setores["CD_UF"] == 25].copy()

print("Total de setores da Paraíba:", len(setores_pb))
print("Total de municípios:", setores_pb["CD_MUN"].nunique())

print("\nSituações encontradas:")
print(setores_pb["SITUACAO"].value_counts(dropna=False))

setores_pb[["CD_MUN", "NM_MUN", "SITUACAO", "v0001"]].head(10)

Total de setores da Paraíba: 9639
Total de municípios: 223

Situações encontradas:
SITUACAO
Urbana    6434
Rural     3200
NaN          5
Name: count, dtype: int64


,CD_MUN,NM_MUN,SITUACAO,v0001
91221,2500106,Água Branca,Urbana,531
91222,2500106,Água Branca,Urbana,430
91223,2500106,Água Branca,Urbana,565
91224,2500106,Água Branca,Rural,284
91225,2500106,Água Branca,Rural,354
91226,2500106,Água Branca,Rural,429
91227,2500106,Água Branca,Rural,368
91228,2500106,Água Branca,Rural,379
91229,2500106,Água Branca,Rural,573
91230,2500106,Água Branca,Rural,248


In [55]:
# Somar a população de cada município por situação urbana/rural
pop_situacao = (
    setores_pb
    .groupby(["CD_MUN", "NM_MUN", "SITUACAO"])["v0001"]
    .sum()
    .reset_index()
)

print(pop_situacao.head(10))
print("\nValores encontrados em SITUACAO:")
print(pop_situacao["SITUACAO"].unique())

    CD_MUN         NM_MUN SITUACAO  v0001
0  2500106    Água Branca    Rural   4841
1  2500106    Água Branca   Urbana   4494
2  2500205         Aguiar    Rural   2154
3  2500205         Aguiar   Urbana   2849
4  2500304  Alagoa Grande    Rural   9332
5  2500304  Alagoa Grande   Urbana  16730
6  2500403    Alagoa Nova    Rural   9428
7  2500403    Alagoa Nova   Urbana  11585
8  2500502      Alagoinha    Rural   4159
9  2500502      Alagoinha   Urbana   9566

Valores encontrados em SITUACAO:
['Rural' 'Urbana']


In [56]:
print(type(pop_situacao))
print(pop_situacao.shape)
pop_situacao.head()

<class 'pandas.core.frame.DataFrame'>
(446, 4)


,CD_MUN,NM_MUN,SITUACAO,v0001
0,2500106,Água Branca,Rural,4841
1,2500106,Água Branca,Urbana,4494
2,2500205,Aguiar,Rural,2154
3,2500205,Aguiar,Urbana,2849
4,2500304,Alagoa Grande,Rural,9332


In [57]:
# Transformar Rural e Urbana em colunas
pop_municipios = pop_situacao.pivot(
    index=["CD_MUN", "NM_MUN"],
    columns="SITUACAO",
    values="v0001"
).reset_index()

# Remover o nome do índice das colunas
pop_municipios.columns.name = None

# Calcular população total
pop_municipios["populacao_total"] = (
    pop_municipios["Rural"] + pop_municipios["Urbana"]
)

# Calcular percentual da população rural
pop_municipios["percentual_rural"] = (
    pop_municipios["Rural"] / pop_municipios["populacao_total"] * 100
).round(2)

# Renomear para nomes mais claros
pop_municipios = pop_municipios.rename(columns={
    "CD_MUN": "codigo_ibge",
    "NM_MUN": "municipio",
    "Rural": "populacao_rural",
    "Urbana": "populacao_urbana"
})

print("Total de municípios:", len(pop_municipios))

pop_municipios.head(10)

Total de municípios: 223


,codigo_ibge,municipio,populacao_rural,populacao_urbana,populacao_total,percentual_rural
0,2500106,Água Branca,4841,4494,9335,51.86
1,2500205,Aguiar,2154,2849,5003,43.05
2,2500304,Alagoa Grande,9332,16730,26062,35.81
3,2500403,Alagoa Nova,9428,11585,21013,44.87
4,2500502,Alagoinha,4159,9566,13725,30.30
5,2500536,Alcantil,2956,2622,5578,52.99
6,2500577,Algodão de Jandaíra,1210,1743,2953,40.98
7,2500601,Alhandra,3767,17963,21730,17.34
8,2500700,São João do Rio do Peixe,10127,7837,17964,56.37
9,2500734,Amparo,947,1287,2234,42.39


In [58]:
# Salvar a base tratada
nome_arquivo = "populacao_urbana_rural_municipios_pb_2022.csv"

pop_municipios.to_csv(
    nome_arquivo,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo com sucesso!")
print(nome_arquivo)
print("Total de municípios:", len(pop_municipios))

Arquivo salvo com sucesso!
populacao_urbana_rural_municipios_pb_2022.csv
Total de municípios: 223


In [59]:
import os

print("Pasta atual:")
print(os.getcwd())

print("\nArquivos nessa pasta:")
print(os.listdir())

Pasta atual:
/content

Arquivos nessa pasta:
['.config', '2_1_Populacao_residente_Area_territorial_Densidade_demografica_xlsx.zip', 'populacao_municipios_pb_2022.csv', 'populacao_urbana_rural_municipios_pb_2022.csv', 'Agregados_por_setores_basico_BR_20260520.zip', 'dados_ibge', 'sample_data']


In [60]:
import pandas as pd
import zipfile
import os

arquivo_setores = "/content/Agregados_por_setores_basico_BR_20260520.zip"

with zipfile.ZipFile(arquivo_setores, "r") as zip_ref:
    print("Arquivos dentro do ZIP:")
    for nome in zip_ref.namelist():
        print(nome)

Arquivos dentro do ZIP:
Agregados_por_setores_basico_BR.csv


In [61]:
import os

arquivo = "/content/Agregados_por_setores_basico_BR_20260520.zip"

print("Existe:", os.path.exists(arquivo))

if os.path.exists(arquivo):
    print("Tamanho:", os.path.getsize(arquivo), "bytes")

    with open(arquivo, "rb") as f:
        inicio = f.read(20)

    print("Primeiros bytes:", inicio)

Existe: True
Tamanho: 15418417 bytes
Primeiros bytes: b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xa5f\xa8\\\x0f+\xd6,\x89C'


In [62]:
import zipfile

arquivo = "/content/Agregados_por_setores_basico_BR_20260520.zip"

print("É reconhecido como ZIP?", zipfile.is_zipfile(arquivo))

try:
    with zipfile.ZipFile(arquivo, "r") as z:
        print("Teste de integridade:", z.testzip())
        print("Quantidade de arquivos:", len(z.namelist()))
except Exception as e:
    print("Erro:", type(e).__name__, "-", e)

É reconhecido como ZIP? True
Teste de integridade: None
Quantidade de arquivos: 1


In [63]:
import pandas as pd
import zipfile

arquivo_setores = "/content/Agregados_por_setores_basico_BR_20260520.zip"

with zipfile.ZipFile(arquivo_setores, "r") as zip_ref:
    nome_csv = "Agregados_por_setores_basico_BR.csv"

    with zip_ref.open(nome_csv) as arquivo:
        setores = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            low_memory=False
        )

print("Total de registros no Brasil:", len(setores))

print("\nColunas encontradas:")
print(setores.columns.tolist())

Total de registros no Brasil: 468099

Colunas encontradas:
['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'v0001', 'v0002', 'v0003', 'v0004', 'v0005', 'v0006', 'v0007', 'v0008', 'v0009']


In [64]:
# Filtrar somente os setores censitários da Paraíba
setores_pb = setores[setores["CD_UF"] == 25].copy()

print("Total de setores da Paraíba:", len(setores_pb))
print("Total de municípios:", setores_pb["CD_MUN"].nunique())

print("\nSituações encontradas:")
print(setores_pb["SITUACAO"].value_counts(dropna=False))

setores_pb[["CD_MUN", "NM_MUN", "SITUACAO", "v0001"]].head(10)

Total de setores da Paraíba: 9639
Total de municípios: 223

Situações encontradas:
SITUACAO
Urbana    6434
Rural     3200
NaN          5
Name: count, dtype: int64


,CD_MUN,NM_MUN,SITUACAO,v0001
91221,2500106,Água Branca,Urbana,531
91222,2500106,Água Branca,Urbana,430
91223,2500106,Água Branca,Urbana,565
91224,2500106,Água Branca,Rural,284
91225,2500106,Água Branca,Rural,354
91226,2500106,Água Branca,Rural,429
91227,2500106,Água Branca,Rural,368
91228,2500106,Água Branca,Rural,379
91229,2500106,Água Branca,Rural,573
91230,2500106,Água Branca,Rural,248


In [65]:
# Conferir o arquivo final antes de enviar para o GitHub
base_final = pd.read_csv(
    "populacao_urbana_rural_municipios_pb_2022.csv"
)

print("Dimensão:", base_final.shape)

print("\nColunas:")
print(base_final.columns.tolist())

print("\nValores ausentes:")
print(base_final.isnull().sum())

base_final.head(10)

Dimensão: (223, 6)

Colunas:
['codigo_ibge', 'municipio', 'populacao_rural', 'populacao_urbana', 'populacao_total', 'percentual_rural']

Valores ausentes:
codigo_ibge         0
municipio           0
populacao_rural     0
populacao_urbana    0
populacao_total     0
percentual_rural    0
dtype: int64


,codigo_ibge,municipio,populacao_rural,populacao_urbana,populacao_total,percentual_rural
0,2500106,Água Branca,4841,4494,9335,51.86
1,2500205,Aguiar,2154,2849,5003,43.05
2,2500304,Alagoa Grande,9332,16730,26062,35.81
3,2500403,Alagoa Nova,9428,11585,21013,44.87
4,2500502,Alagoinha,4159,9566,13725,30.30
5,2500536,Alcantil,2956,2622,5578,52.99
6,2500577,Algodão de Jandaíra,1210,1743,2953,40.98
7,2500601,Alhandra,3767,17963,21730,17.34
8,2500700,São João do Rio do Peixe,10127,7837,17964,56.37
9,2500734,Amparo,947,1287,2234,42.39


In [66]:
from google.colab import files

files.download("populacao_urbana_rural_municipios_pb_2022.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resultado da aquisição e tratamento

Nesta etapa foram adquiridos e tratados dados do Censo Demográfico 2022 do IBGE para os municípios do estado da Paraíba.

A base de Agregados por Setores Censitários foi filtrada utilizando o código da UF da Paraíba (`CD_UF = 25`). Após a filtragem, foram identificados 9.639 setores censitários pertencentes aos 223 municípios do estado.

Os setores foram agrupados de acordo com o município e sua situação (`Urbana` ou `Rural`). A partir desse processamento, foram calculadas a população urbana, a população rural, a população total e o percentual da população rural de cada município.

A base final possui:

- 223 municípios;
- 6 variáveis;
- nenhum valor ausente nas variáveis utilizadas.

O arquivo processado gerado é:

`populacao_urbana_rural_municipios_pb_2022.csv`

As principais variáveis da base final são:

- `codigo_ibge`: código oficial do município;
- `municipio`: nome do município;
- `populacao_rural`: população residente em áreas rurais;
- `populacao_urbana`: população residente em áreas urbanas;
- `populacao_total`: soma das populações rural e urbana;
- `percentual_rural`: participação percentual da população rural no total do município.

Esse arquivo poderá ser utilizado posteriormente para integração com outras bases do projeto por meio do código IBGE do município.